# Installing Libraries

The Required Libraries for the current process is loaded into the system by executing the following commands in a seperate python virtual environment's terminal:

```bash
pip install numpy
pip install pandas
```

The rest of the Libraries are already included in the python installation by default

# Importing Packages

In [1]:
import pandas as pd
import numpy as np
import os

# Loading and Filtering Data

We load the Metadata for the Dataset and the cleaned Subsets for the Train/Val and Test Splits and clean the Metadata based on the Lists

In [2]:
DATASET_PATH = "Dataset"
labels = pd.read_csv(os.path.join(DATASET_PATH, "01. labels.csv"))

In [3]:
print("Original metadata size:", len(labels))
labels.head()

Original metadata size: 112120


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [4]:
with open(os.path.join(DATASET_PATH, "02. train_val_subset.txt")) as f:
    train_val_subset = set(f.read().splitlines())

with open(os.path.join(DATASET_PATH, "02. test_subset.txt")) as f:
    test_subset = set(f.read().splitlines())

In [5]:
subset_images = train_val_subset.union(test_subset)
print("Subset images:", len(subset_images))

Subset images: 14999


In [6]:
subset_df = labels[labels["Image Index"].isin(subset_images)].copy()
print("Subset metadata size:", len(subset_df))

Subset metadata size: 14999


# Verify Images

Once again, verify if all images are in the `Dataset/Images` Directory

In [7]:
IMAGE_PATH = os.path.join(DATASET_PATH, "Images")

subset_df["image_exists"] = subset_df["Image Index"].apply(
    lambda x: os.path.exists(os.path.join(IMAGE_PATH, x))
)

In [8]:
missing_files = subset_df[subset_df["image_exists"] == False]

print("Missing image files:", len(missing_files))
subset_df.drop(columns=["image_exists"], inplace=True)

Missing image files: 0


# Class Label Examinaion

Now we Examine the Class Labels of the Dataset to know what Diseases and Conditions are included in the Dataset

In [9]:
subset_df["Finding Labels"].head()

0              Cardiomegaly
1    Cardiomegaly|Emphysema
2     Cardiomegaly|Effusion
3                No Finding
4                    Hernia
Name: Finding Labels, dtype: str

In [10]:
all_labels = set()

for labels_str in subset_df["Finding Labels"]:
    for disease in labels_str.split("|"):
        all_labels.add(disease)

sorted(all_labels)

['Atelectasis',
 'Cardiomegaly',
 'Consolidation',
 'Edema',
 'Effusion',
 'Emphysema',
 'Fibrosis',
 'Hernia',
 'Infiltration',
 'Mass',
 'No Finding',
 'Nodule',
 'Pleural_Thickening',
 'Pneumonia',
 'Pneumothorax']

# Multi One-Hot Encoding

Since Deep Learning models require binary columns per disease, and we have multiple diseases with same entry, we now perform **Multi One-Hot Encoding** to the Dataset so that we can work with it.

In [11]:
diseases = sorted(all_labels)
diseases

['Atelectasis',
 'Cardiomegaly',
 'Consolidation',
 'Edema',
 'Effusion',
 'Emphysema',
 'Fibrosis',
 'Hernia',
 'Infiltration',
 'Mass',
 'No Finding',
 'Nodule',
 'Pleural_Thickening',
 'Pneumonia',
 'Pneumothorax']

In [12]:
for disease in diseases:
    subset_df[disease] = subset_df["Finding Labels"].apply(
        lambda x: 1 if disease in x else 0
    )

In [13]:
subset_df

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,...,Emphysema,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,...,0,0,0,0,0,0,0,0,0,0
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,...,1,0,0,0,0,0,0,0,0,0
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,...,0,0,0,0,0,0,0,0,0,0
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,...,0,0,0,0,0,1,0,0,0,0
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14994,00003923_009.png,Effusion|Infiltration,9,3923,28,M,AP,2500,2048,0.171,...,0,0,0,1,0,0,0,0,0,0
14995,00003923_010.png,Effusion|Infiltration,10,3923,28,M,AP,2500,2048,0.171,...,0,0,0,1,0,0,0,0,0,0
14996,00003923_011.png,Infiltration,11,3923,28,M,AP,2500,2048,0.171,...,0,0,0,1,0,0,0,0,0,0
14997,00003923_012.png,No Finding,12,3923,28,M,AP,2500,2048,0.171,...,0,0,0,0,0,1,0,0,0,0


# Class Distribution

Now we check the Class Distribution of the Dataset for any class imbalances and skewness

In [14]:
disease_counts = subset_df[diseases].sum().sort_values(ascending=False)
print(disease_counts)

No Finding            8753
Infiltration          2260
Effusion              1382
Atelectasis           1350
Pneumothorax           670
Nodule                 666
Consolidation          557
Mass                   470
Pleural_Thickening     468
Cardiomegaly           396
Fibrosis               379
Emphysema              318
Edema                  196
Pneumonia              184
Hernia                  46
dtype: int64


There exists an heavy skewness to the Dataset, which is a noted characteristic of [NIH Chest x-Rays](https://www.kaggle.com/datasets/nih-chest-xrays/data) Dataset, which is common in real-life scenario. So we need our model to learn the real distribution. To Handle it, we would use `class-weighted loss` at the time of training. One such workflow is given below:
```txt
Dataset
   ↓
DataLoader
   ↓
Model (DenseNet / ResNet)
   ↓
Weighted BCE Loss
   ↓
Optimizer
   ↓
AUROC evaluation
```

# Dataset Split

We now assign the Train/Val or Test Split tags for each Image's Metadata, so that it would be easier to split data in future

In [15]:
def assign_split(image):
    if image in train_val_subset:
        return "train_val"
    else:
        return "test"

subset_df["split"] = subset_df["Image Index"].apply(assign_split)
subset_df["split"].value_counts()

split
train_val    12540
test          2459
Name: count, dtype: int64

In [16]:
subset_df

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,...,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,split
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,...,0,0,0,0,0,0,0,0,0,train_val
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,...,0,0,0,0,0,0,0,0,0,train_val
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,...,0,0,0,0,0,0,0,0,0,train_val
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,...,0,0,0,0,1,0,0,0,0,train_val
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,...,0,1,0,0,0,0,0,0,0,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14994,00003923_009.png,Effusion|Infiltration,9,3923,28,M,AP,2500,2048,0.171,...,0,0,1,0,0,0,0,0,0,train_val
14995,00003923_010.png,Effusion|Infiltration,10,3923,28,M,AP,2500,2048,0.171,...,0,0,1,0,0,0,0,0,0,train_val
14996,00003923_011.png,Infiltration,11,3923,28,M,AP,2500,2048,0.171,...,0,0,1,0,0,0,0,0,0,train_val
14997,00003923_012.png,No Finding,12,3923,28,M,AP,2500,2048,0.171,...,0,0,0,0,1,0,0,0,0,train_val


# Exporting Metadata

Now we export the cleaned metadata so that it can be used later on

In [17]:
clean_path = os.path.join(DATASET_PATH, "02. cleaned_dataset.csv")

subset_df.to_csv(clean_path, index=False)